# Lab 04: How to find the stage where a RAG answer failed

**CPU | API key: `OPENAI_API_KEY` (chat and embedding calls) | In class**

## Goal

Follow every question through the pipeline, find the wrong answers, and name the step where each one went wrong. The
moment that teaches it: the wrong answers all fail at the same step, after the right passage was found and given to
the model. Giving the model that passage alone shows three different causes: one answer was misled by the other
passages, one was right all along in words the answer key did not list, and one the model still refuses. One failing
step, three different repairs.

## What you will learn

- What a trace record holds, and how to make one for a question by hand.
- How to trace a whole question set and count what it cost.
- How to screen answers against expected answer strings, and read the answers the screen flags.
- How to build a stage table that follows the supporting passage through the hits, the pack and the answer.
- How to use oracle context to tell a failure caused by the other evidence from one caused by the answer step, and
  how an incomplete label turns a right answer into a "miss".
- How to save a baseline run that later labs compare against.

**In class:** Parts A and D. Parts B, C, E and F are self-study. Run the cells of Parts B and C in class without
stopping on them, because Part D reads the traces and screen results they make.

## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-18
OPENAI_API_KEY    found in .env


This lab uses `chat_model` and `embed_model` from the lines above. It makes about 90 short chat calls, well under one
US cent on `gpt-4o-mini`. The saved outputs and every count in the text come from `gpt-4o-mini`; another model can
fail on other questions (the README lists what `gpt-6-luna` did), so read your own outputs against the text: model replies vary from run to run and similarity scores can move in the third decimal, so compare the order of results and the kind of reply with the text, not every number.

---

## Part A: Trace one question by hand

Part A records what each step of the pipeline produced for one question, so that a wrong answer can be read step by
step afterwards. That record is a **trace**.

The pipeline is Lab 02's: search the 272 chunks and the five policies, pack the best hits as `[ID] text` lines within
400 tokens, and ask for a cited answer as JSON. The trace keeps one thing per step:

| Step | What the trace keeps | The question it answers when an answer is wrong |
|---|---|---|
| Search | the hits: chunk IDs and scores | Was the right passage found, and how high? |
| Packing | the IDs of the packed lines | Did it reach the prompt? |
| Prompt | a hash of the messages | Was this the same prompt as in another run? |
| Answer | the raw reply and the tokens billed | What did the model say, and what did it cost? |
| Run | the stamp | Which models, which endpoint, which day? |

Start with the corpus and its cached vectors, as in Lab 02.

In [3]:
import json
from shared.search import chunk_vectors, search
from shared.answer import RULES, pack, parse, response_format, words

DATA = ROOT / "data"
chunks = json.loads((DATA / "squad-chunks.json").read_text(encoding="utf-8"))
policies = json.loads((DATA / "support-case.json").read_text(encoding="utf-8"))
corpus = chunks + [{**policy, "chunk_id": policy["source_id"]} for policy in policies]
vectors = chunk_vectors(corpus, settings)
print(len(corpus), "chunks |", vectors.shape)

277 chunks | (277, 1536)


Search and pack the Tesla question, and print what the two steps returned.

In [4]:
question = "In what year was Nikola Tesla born?"
hits = search(question, corpus, vectors, settings, k=5)
lines = pack(hits, settings)
print("hits:  ", [(chunk["chunk_id"], round(score, 4)) for score, chunk in hits])
print("packed:", [line[1:line.index("]")] for line in lines])

hits:   [('S03-P00-C00', 0.722), ('S03-P01-C00', 0.5404), ('S03-P02-C00', 0.4664), ('S03-P01-C01', 0.2592), ('S06-P00-C00', 0.25)]
packed: ['S03-P00-C00', 'S03-P01-C00', 'S03-P02-C00', 'S03-P01-C01']


Five hits, four packed: the fifth hit, from another article, would have passed the 400-token
budget. The trace keeps the scores too, so it shows how close a chunk came, not only whether it was packed.

Next, the prompt: the rules from `shared/answer.py` as the system message, and the question with the evidence as the
user message, exactly as `answer()` builds them. A **hash** is a short fingerprint of a text: the same text always
gives the same hash, and any change, even one character, gives a different one. The trace keeps the hash instead of
the whole prompt: two traces with the same hash had the same prompt, word for word.

In [5]:
import hashlib

user = f"Question: {question}\nEvidence:\n" + "\n".join(lines)
messages = [{"role": "system", "content": RULES}, {"role": "user", "content": user}]
prompt_sha = hashlib.sha256(json.dumps(messages).encode()).hexdigest()[:12]
print(prompt_sha)

c2aa8c003db7


Then the call `answer()` makes, with one difference: keep the reply as text, without parsing it. If a reply is
broken, the trace should show how.

In [6]:
from shared.settings import chat_client, chat_options, chat_url

reply = chat_client(settings).chat.completions.create(
    **chat_options("CHAT_MODEL", settings), response_format=response_format(settings), messages=messages)
print(reply.choices[0].message.content)
print("tokens billed:", reply.usage.prompt_tokens, "in,", reply.usage.completion_tokens, "out")

{"answer":"Nikola Tesla was born in the year 1856.","claims":[{"claim":"Nikola Tesla was born in the year 1856.","source_id":"S03-P00-C00","quote":"10 July 1856"}]}
tokens billed: 500 in, 51 out


Put the pieces in one dictionary, with the stamp from Config, and print it field by field. This is the trace record.

In [7]:
record = {
    "question": question,
    "hits": [[chunk["chunk_id"], round(score, 4)] for score, chunk in hits],
    "packed": [line[1:line.index("]")] for line in lines],
    "prompt_sha": prompt_sha,
    "reply": reply.choices[0].message.content,
    "usage": {"input": reply.usage.prompt_tokens, "output": reply.usage.completion_tokens},
    "stamp": stamp(settings),
}
for name, value in record.items():
    print(f"{name:11}{value}")

question   In what year was Nikola Tesla born?
hits       [['S03-P00-C00', 0.722], ['S03-P01-C00', 0.5404], ['S03-P02-C00', 0.4664], ['S03-P01-C01', 0.2592], ['S06-P00-C00', 0.25]]
packed     ['S03-P00-C00', 'S03-P01-C00', 'S03-P02-C00', 'S03-P01-C01']
prompt_sha c2aa8c003db7
reply      {"answer":"Nikola Tesla was born in the year 1856.","claims":[{"claim":"Nikola Tesla was born in the year 1856.","source_id":"S03-P00-C00","quote":"10 July 1856"}]}
usage      {'input': 500, 'output': 51}
stamp      {'chat_model': 'gpt-4o-mini', 'reasoning_effort': None, 'embed_model': 'text-embedding-3-small', 'judge_model': 'gpt-4o-mini', 'rerank_model': 'gpt-4o-mini', 'chat_endpoint': 'https://api.openai.com/v1', 'embed_endpoint': 'https://api.openai.com/v1', 'run_date': '2026-10-18'}


One record holds the whole run of one question. The reply is the raw JSON text: it can be parsed later,
or read as it came if it is broken. `usage` shows 500 input tokens billed, against 353 for Lab 02's first plain call:
the rules now ask for claims, quotes and JSON. The stamp names the models, the endpoint and the day.

### Checkpoint A

| You should see | Our run |
|---|---|
| Corpus | 277 chunks, vectors `(277, 1536)` |
| Tesla's trace | 5 hits, `S03-P00-C00` first (about 0.72); 4 packed; `prompt_sha` `c2aa8c003db7` |
| The reply | JSON with *Nikola Tesla was born in the year 1856.* (or *on 10 July 1856*) and one claim from `S03-P00-C00`; 500 input tokens |

---

## Part B: Trace every question (self-study)

Part B turns Part A into a function and runs it on every question, so that each wrong answer comes with its record.

`trace()` is Part A's cells joined, nothing added. Check it on the Tesla question: the prompt hash must match Part A's.

In [8]:
def trace(question, k=5):
    hits = search(question, corpus, vectors, settings, k=k)
    lines = pack(hits, settings)
    user = f"Question: {question}\nEvidence:\n" + "\n".join(lines)
    messages = [{"role": "system", "content": RULES}, {"role": "user", "content": user}]
    reply = chat_client(settings).chat.completions.create(
        **chat_options("CHAT_MODEL", settings), response_format=response_format(settings), messages=messages)
    return {"question": question,
            "hits": [[chunk["chunk_id"], round(score, 4)] for score, chunk in hits],
            "packed": [line[1:line.index("]")] for line in lines],
            "prompt_sha": hashlib.sha256(json.dumps(messages).encode()).hexdigest()[:12],
            "reply": reply.choices[0].message.content,
            "usage": {"input": reply.usage.prompt_tokens, "output": reply.usage.completion_tokens},
            "stamp": stamp(settings)}

print("same prompt as Part A:", trace(question)["prompt_sha"] == record["prompt_sha"])

same prompt as Part A: True


Now the questions. The 30 development questions each come with SQuAD's answer strings, the words people marked as
the answer in the paragraph, and `known_support`, the paragraph they were written from. Add four questions about Atlas,
written for this lab, with the answer strings the policies give and the policy that holds the answer. Every question
gets the same three fields: the question, the `expected` answer strings and its `support`.

In [9]:
dev = json.loads((DATA / "development-questions.json").read_text(encoding="utf-8"))
questions = [{"question": q["question"], "expected": sorted({a["text"] for a in q["answers"]}),
              "support": q["known_support"]} for q in dev]
questions += [  # written for this lab: the answers the Atlas policies give
    {"question": "How long does an Atlas reset link remain valid?", "expected": ["30 minutes"], "support": ["H02"]},
    {"question": "Who can export all reports in Atlas?", "expected": ["workspace owners"], "support": ["H03"]},
    {"question": "Can temporary Atlas accounts reset by email?", "expected": ["support agent"], "support": ["H01"]},
    {"question": "Do I need the mobile app to reset my Atlas password?", "expected": ["no", "by email"], "support": ["H01"]},
]
print(len(questions), "questions")

34 questions


Trace all of them, one call each, and add up the tokens from the records.

In [10]:
traces = [trace(q["question"]) for q in questions]
print(len(traces), "traces |", sum(t["usage"]["input"] for t in traces), "input tokens,",
      sum(t["usage"]["output"] for t in traces), "output tokens")

34 traces | 17641 input tokens, 2301 output tokens


The function builds the same prompt as the hand version. The 34 traces cost 17,641 input tokens and 2,301
output tokens, added up from the records: a trace is also the bill.

### Checkpoint B

| You should see | Our run |
|---|---|
| `trace()` on Tesla | `same prompt as Part A: True` |
| All questions | 34 traces; 17,641 input tokens; output tokens vary by run (2,301 in ours; 2,240 to 2,445 over our runs) |

---

## Part C: Screen the answers (self-study)

Part C picks out the traces worth reading. A **screen** is a quick check in code that flags answers for a person to
read; it does not grade them.

This one passes an answer when one of the question's expected strings appears in it. It compares whole words in lower
case, with Lab 02's `words()`, and adds a space at each end, so that *no* is not found inside *know* or *not*.

In [11]:
def screen(answer_text, expected):
    answer = f" {words(answer_text)} "
    return any(f" {words(text)} " in answer for text in expected)

answers = [parse(t["reply"])["answer"] for t in traces]
passed = [screen(a, q["expected"]) for a, q in zip(answers, questions)]
print(f"passed {sum(passed)} of {len(passed)}")
for a, q, ok in zip(answers, questions, passed):
    if not ok:
        print(f"\n{q['question']}\n  expected: {q['expected']}\n  answer:   {a}")

passed 31 of 34

What measure of a computational problem broadly defines the inherent difficulty of the solution?
  expected: ['if its solution requires significant resources', 'its solution requires significant resources']
  answer:   Computational complexity theory broadly defines the inherent difficulty of a computational problem.

Can temporary Atlas accounts reset by email?
  expected: ['support agent']
  answer:   Insufficient evidence.

Do I need the mobile app to reset my Atlas password?
  expected: ['no', 'by email']
  answer:   Yes, password reset uses a mobile app.


31 of 34 pass. Read the three flagged answers before deciding anything:

- The complexity question gets the name of the field, *computational complexity theory*. The label asks for what
  makes a problem hard: *if its solution requires significant resources*.
- The temporary-accounts question is refused, though H01 says temporary accounts *require a support agent*. Lab 02
  measured the same refusal.
- The mobile-app question gets *Yes, password reset uses a mobile app.* That sentence is in H04, the Boreal policy;
  Atlas passwords are reset by email (H01).

All three answers are wrong.

A pass is not proof either: the screen checks that the words are there, not that the evidence supports them. Lab 02's
quote check and Lab 07's judge read more; the screen only decides what to read first.

### Checkpoint C

| You should see | Our run |
|---|---|
| The screen | `passed 31 of 34` |
| Flagged | the complexity question, the temporary-accounts question and the mobile-app question |

---

## Part D: Find the stage where the support was lost

Part D asks where each answer lost its **support**, the passage that holds the answer: the paragraph in
`known_support` (`S04-P01`), or the policy for the Atlas questions. Follow it through the pipeline in order. The
first step it does not reach names the stage to inspect:

| The support is ... | Stage | What to inspect next |
|---|---|---|
| not in the corpus | corpus | whether the document was loaded at all |
| in the corpus, not in the hits | retrieval | the chunks, the question's wording, the ranking |
| in the hits, not in the pack | packing | the token budget, the order, any filter |
| in the pack, and the answer is wrong | generation | the rules and what the model did with the evidence |

The hits and the pack hold chunk IDs (`S04-P01-C00`), and the support is a paragraph ID (`S04-P01`), so each chunk ID
is mapped to its paragraph first. Before you run the cell, write down where you expect the three flagged answers to
have lost their support.

In [12]:
source_of = {chunk["chunk_id"]: chunk["source_id"] for chunk in corpus}

def stages(t, q, ok):
    return {"corpus": any(chunk["source_id"] in q["support"] for chunk in corpus),
            "hits": any(source_of[chunk_id] in q["support"] for chunk_id, _ in t["hits"]),
            "pack": any(source_of[chunk_id] in q["support"] for chunk_id in t["packed"]),
            "answer": ok}

table = [stages(t, q, ok) for t, q, ok in zip(traces, questions, passed)]
for step in ("corpus", "hits", "pack"):
    print(f"support in the {step:7} {sum(row[step] for row in table)} of {len(table)}")
print(f"answer passes screen {sum(row['answer'] for row in table)} of {len(table)}")
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)}")

support in the corpus  34 of 34
support in the hits    34 of 34
support in the pack    34 of 34
answer passes screen 31 of 34

gpt-4o-mini at https://api.openai.com/v1


The support reached the corpus, the hits and the pack for all 34 questions; nothing was lost before the
answer step. Name the stage for each flagged trace, and print where its support ranked:

In [13]:
STAGE = {"corpus": "corpus", "hits": "retrieval", "pack": "packing", "answer": "generation"}

for t, q, row in zip(traces, questions, table):
    lost = next((step for step in row if not row[step]), None)
    if lost:
        ranks = [i + 1 for i, (chunk_id, _) in enumerate(t["hits"]) if source_of[chunk_id] in q["support"]]
        print(f"{STAGE[lost]:10} | support at hit {ranks} | hits {[c for c, _ in t['hits']]} | {q['question']}")

generation | support at hit [1, 4] | hits ['S04-P01-C00', 'S04-P00-C00', 'S04-P02-C01', 'S04-P01-C01', 'S04-P02-C00'] | What measure of a computational problem broadly defines the inherent difficulty of the solution?
generation | support at hit [1] | hits ['H01', 'H04', 'H02', 'H05', 'H03'] | Can temporary Atlas accounts reset by email?
generation | support at hit [2] | hits ['H04', 'H01', 'H02', 'H05', 'H03'] | Do I need the mobile app to reset my Atlas password?


All three stop at generation: the support was packed, and for two of them it was the first hit. On the
mobile-app question H04, the Boreal policy, ranked first and H01 second. The complexity question's paragraph,
`S04-P01`, is in the pack twice, as its two chunks at hits 1 and 4.

The stage table cannot say more. It knows that the support reached the prompt; it does not know what the rest of the
prompt did to the answer.

### Checkpoint D

| You should see | Our run |
|---|---|
| Support | corpus, hits and pack 34 of 34 each; `answer passes screen 31 of 34` |
| Stages | `generation` for all three; support at hits 1 and 4 (complexity), 1 (temporary accounts), 2 (mobile app) |

---

## Part E: Give the model only the labelled passage (self-study)

Part E tests the answer step on its own. **Oracle context** replaces the retrieved evidence with the passage the label
names, so that search and packing cannot be the cause. It is a diagnostic only: a real user's question comes with no
label, so an oracle run is never a result to report.

Read the outcome in two ways. If the answer becomes right, the model can answer from the support, and the trouble is
in what else the pipeline put next to it. If the answer stays wrong, the trouble is in the answer step itself: the
rules or the model. Collect the passage texts and the three flagged questions:

In [14]:
passages = {p["source_id"]: p["text"] for p in json.loads((DATA / "squad-passages.json").read_text(encoding="utf-8"))}
passages.update({p["source_id"]: p["text"] for p in policies})

failed = [q for q, ok in zip(questions, passed) if not ok]
for q in failed:
    print(q["support"], "|", passages[q["support"][0]][:110], "...")

['S04-P01'] | A problem is regarded as inherently difficult if its solution requires significant resources, whatever the alg ...
['H01'] | Atlas passwords can be reset by email. Temporary accounts require a support agent. ...
['H01'] | Atlas passwords can be reset by email. Temporary accounts require a support agent. ...


Ask each flagged question five times with only its support as evidence, through `answer()` from `shared/answer.py`
(the same rules and schema), and screen every answer.

In [15]:
from collections import Counter
from shared.answer import answer

for q in failed:
    oracle = [f"[{source_id}] {passages[source_id]}" for source_id in q["support"]]
    oracle_answers = [answer(q["question"], oracle, settings)["answer"] for _ in range(5)]
    print(f"\n{q['question']} | passes the screen {sum(screen(a, q['expected']) for a in oracle_answers)} of 5")
    for text, n in Counter(oracle_answers).most_common():
        print(f"  {n} of 5 | {text}")


What measure of a computational problem broadly defines the inherent difficulty of the solution? | passes the screen 0 of 5
  4 of 5 | The measure of a computational problem that broadly defines the inherent difficulty of the solution is the amount of resources needed to solve them, such as time and storage.
  1 of 5 | The measure of a computational problem that broadly defines the inherent difficulty of the solution is the quantification of the amount of resources needed to solve them, such as time and storage.



Can temporary Atlas accounts reset by email? | passes the screen 0 of 5
  5 of 5 | Insufficient evidence.



Do I need the mobile app to reset my Atlas password? | passes the screen 5 of 5
  5 of 5 | No, you do not need the mobile app to reset your Atlas password.


Three different results.

The mobile-app question, with H01 alone, gets *No, you do not need the mobile app to reset your Atlas password.* in
5 of 5 runs, and the screen passes it. The model can answer from H01; in the pipeline, another packed line misled
it. H04's second sentence, *Password reset uses a mobile app.*, names no product, and H04 ranked first.

The complexity question now gets *the amount of resources needed to solve them, such as time and storage* in all
five runs (one adds *the quantification of*), and the screen fails all five. Read the passage: the theory works by
*quantifying the amount of resources needed to solve them, such as time and storage*. The answer is right, in other words than the label's. SQuAD's label
lists the words people marked, not every right wording, so a screen built on it calls a right answer a miss: the
label is **incomplete**. In the pipeline, the other S04 chunks, about classifying problems by *inherent difficulty*,
pulled the answer to the name of the field.

The temporary-accounts question is still *Insufficient evidence.* in 5 of 5, with only the passage that holds the
answer. Search and packing are not the cause. Take the abstention line out of the rules and ask again from H01 alone:

In [16]:
ABSTAIN = ' If the evidence does not answer the question, the answer is "Insufficient evidence." and the claims list is empty.'
rules_without = RULES.replace(ABSTAIN, "")
temporary = next(q for q in questions if "temporary" in q["question"])

print("abstention line removed:", rules_without != RULES)
oracle_answers = [answer(temporary["question"], [f"[H01] {passages['H01']}"], settings, rules=rules_without)["answer"]
                  for _ in range(5)]
for text, n in Counter(oracle_answers).most_common():
    print(f"  {n} of 5 | {text}")

abstention line removed: True


  5 of 5 | Temporary Atlas accounts cannot reset by email; they require a support agent.


Without the line, 5 of 5 answers say temporary accounts cannot reset by email and require a support
agent. The abstention line causes this refusal: H01 states the rule, the answer takes one step of reasoning from it,
and with the line the model refuses to take that step.

| Question | Stage table | Oracle context | Where to look next |
|---|---|---|---|
| *Do I need the mobile app to reset my Atlas password?* | generation | right in 5 of 5 | what was packed beside the support: H04, another product's policy, ranked first. Keep other products' policies out of the pack, for example with a metadata filter (Lab 11) |
| *What measure of a computational problem broadly defines ...?* | generation | right in other words in 5 of 5; the screen fails it | the label: accept other wordings, or have a person or a checked judge read it (Lab 07); then the pack, whose other chunks pulled the answer away |
| *Can temporary Atlas accounts reset by email?* | generation | still refused in 5 of 5; answered without the abstention line | the rules: the abstention line refuses an answer that takes one step of reasoning |

All three failed at the same stage in the table, and each needs a different change. Changing the rules would not
remove H04 from the pack, and a better search would not change what the abstention line does. Change the stage the
evidence points to, then rerun the same questions and compare the traces.

### Checkpoint E

| You should see | Our run |
|---|---|
| Mobile app, oracle | *No, you do not need the mobile app ...* in 5 of 5; the screen passes it |
| Complexity, oracle | answers about *the amount of resources needed to solve them* in 4 or 5 of 5 (5 in ours); the screen passes 0 of 5 |
| Temporary accounts, oracle | *Insufficient evidence.* in 5 of 5; without the abstention line, answered in 5 of 5 |

---

## Part F: Save the baseline in `shared/trace.py` (self-study)

Part F keeps the trace step for the labs after this one and saves one traced run of the 30 development questions,
the **baseline**. When a later lab changes a stage, it traces the same questions again and compares them with the
baseline, question by question.

`shared/trace.py` holds Part A's corpus as `corpus()`, Part B's function as `trace()`, and `baseline()`, which traces
the 30 development questions once and saves the records in `.cache/`. The file's name holds the chat model and a hash
of the settings, the rules and the questions, as Lab 01's vector cache does, so a run with another model or endpoint
gets its own file. Read it:

In [17]:
import inspect
import shared.trace

print(inspect.getsource(shared.trace))

"""Trace records for the pipeline: the steps Lab 04 builds by hand, kept for the labs after it.

trace() runs one question through search, packing and the answer call, and returns what each step produced:
the hits, the packed IDs, a hash of the prompt, the raw reply, the tokens billed and the stamp. baseline() traces
the 30 development questions once and saves them in .cache/, in a file named after the chat model and a hash of
the settings, the rules and the questions, so a changed model or endpoint never reuses old traces.
"""
from __future__ import annotations

import hashlib
import json

from shared.answer import RULES, pack, response_format
from shared.search import chunk_vectors, search
from shared.settings import DATA, ROOT, chat_client, chat_options, stamp


def corpus() -> list[dict]:
    """The 272 chunks plus the five policies, each policy as one chunk whose ID is its source ID."""
    chunks = json.loads((DATA / "squad-chunks.json").read_text(encoding="utf-8"))
    policies 

Run `baseline()` twice. The first call traces the 30 questions and saves them; the second reads the file. Then compare
the prompt hashes with Part B's traces of the same questions, and screen the baseline.

In [18]:
from shared.trace import baseline

base = baseline(settings)
print(len(base), "traces | read back the same:", baseline(settings) == base)
print("baseline files:", sorted(path.name for path in (ROOT / ".cache").glob("baseline-*.json")))
print("same prompts as Part B:", sum(b["prompt_sha"] == t["prompt_sha"] for b, t in zip(base, traces)), "of", len(base))
print("passed the screen:", sum(screen(parse(b["reply"])["answer"], q["expected"]) for b, q in zip(base, questions)),
      "of", len(base))
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)}")

30 traces | read back the same: True
baseline files: ['baseline-gpt-4o-mini-c358e23dfda2.json']
same prompts as Part B: 30 of 30
passed the screen: 29 of 30

gpt-4o-mini at https://api.openai.com/v1


The baseline holds 30 traces, and the second call read the same records back from the file. All 30 prompt
hashes match Part B's: the same rules met the same evidence, so any answer that differs between the two runs differs
because of the model, not the pipeline. 29 of 30 pass the screen; the miss is the complexity question. The file's name
carries `gpt-4o-mini`: with another model in `models.env`, `baseline()` traces again into a new file. From here on the labs call `baseline()` and `trace()` by name.

### Checkpoint F

| You should see | Our run |
|---|---|
| The baseline | 30 traces, `read back the same: True`; one file `baseline-gpt-4o-mini-<hash>.json`; same prompts 30 of 30; 29 of 30 pass the screen |

---

## What you can do now

- Make a trace record for one question by hand: hits, packed IDs, prompt hash, raw reply, tokens and stamp.
- Trace a question set with one function and add up its tokens from the records.
- Screen answers against expected strings by whole words, and read every answer the screen flags.
- Build a stage table and name the stage where each answer lost its support.
- Use oracle context to tell whether the other evidence or the answer step caused a failure, and recognise a right
  answer that an incomplete label calls a miss.
- Save and reload a baseline run with `shared/trace.py`, named by model and endpoint.

## Challenge

Use only what this lab showed. A worked solution is in `solved/` for the instructor. Task 1 is the challenge; task 2 is
optional. For each task, keep the printed results and your answer to **Explain**.

1. **Find the line that misled the model.** Oracle context said the support alone gives the right answer for the
   mobile-app question. Which of the other packed lines caused the wrong one?

   - Complete `without_each` under **Task 1**: for each packed line of a trace, answer the question three times with
     that line left out, and print the line's ID and the answers.
   - The cell runs it on the mobile-app question and on the complexity question.

   **Check:** for the mobile-app question, leaving out H04 gives *No, you do not need the mobile app ...* in 3 of 3,
   and leaving out any other line keeps a *Yes*. For the complexity question the result moved between our runs: twice no single line
   changed the answer, and once leaving out `S04-P00-C00`, the chunk about classifying problems by *inherent difficulty*,
   gave the resources answer in 3 of 3.

   **Explain:** what repair does each result point to? Why could a line that is true (H04 is right about Boreal) still
   be the cause?

2. **Optional: name the stage in traces that lost the support earlier.** None of the 34 traces lost its support before
   the answer step. Under **Task 2** are three traces written by hand for *How long does an Atlas reset link remain
   valid?*, in the same shape as `trace()`'s records. Run Part D's `stages()` on each and print the stage.

   **Check:** `['retrieval', 'packing', 'generation']`

   **Explain:** for each trace, what would you change first, and which question would you rerun to check it?

**Task 1.** Complete `without_each`, then run the cell:

In [19]:
def without_each(question):
    """For each packed line: its ID, and three answers to the question from the other lines."""
    lines = pack(search(question, corpus, vectors, settings, k=5), settings)
    # TODO: for each line, answer the question three times from the other lines (answer(question, shorter, settings));
    #       collect (the left-out line's ID, line[1:line.index("]")], with Counter(...) of the three answers)
    return None

for q in [q for q in questions if "mobile app" in q["question"] or "measure" in q["question"]]:
    result = without_each(q["question"])
    if result is None:
        print("⚠️ Complete without_each, then run the cell again.")
        break
    print(q["question"])
    for left_out, counts in result:
        print(f"  without {left_out:12} {counts.most_common()}")

⚠️ Complete without_each, then run the cell again.


**Task 2 (optional).** Three traces written by hand. Print the stage for each:

In [20]:
atlas = {"question": "How long does an Atlas reset link remain valid?", "expected": ["30 minutes"], "support": ["H02"]}
written = [  # written by hand, not from a run
    {"hits": [["H01", 0.61], ["H04", 0.55], ["H03", 0.40]], "packed": ["H01", "H04"],
     "reply": '{"answer": "Atlas reset links last 24 hours.", "claims": []}'},
    {"hits": [["H01", 0.61], ["H02", 0.58], ["H04", 0.55]], "packed": ["H01"],
     "reply": '{"answer": "Atlas reset links last 24 hours.", "claims": []}'},
    {"hits": [["H02", 0.70], ["H01", 0.61], ["H04", 0.55]], "packed": ["H02", "H01"],
     "reply": '{"answer": "Atlas reset links last 24 hours.", "claims": []}'},
]

def stage_of(t, q):
    # TODO: screen the answer in t["reply"], build the row with stages(t, q, ok),
    #       and return STAGE[...] of the first step that is False
    return None

found = [stage_of(t, atlas) for t in written]
print(found if None not in found else "⚠️ Complete stage_of, then run the cell again.")

⚠️ Complete stage_of, then run the cell again.


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| The stage table says `support in the hits 4 of 34` | chunk IDs compared with paragraph IDs directly; only the policies match, because a policy's chunk ID is its source ID | Map each chunk ID through `source_of` first, as `stages()` does |
| `KeyError: 'S04-P01-C00'` in Part E | a chunk ID looked up in `passages`, which is keyed by paragraph ID | Use the question's `support` list, or `source_of[chunk_id]` |
| `JSONDecodeError: Expecting value: line 1 column 1 (char 0)` from `parse(t["reply"])` | the reply in that trace is not JSON (a server that ignored `response_format`); the trace keeps the reply as it came | Read the raw `reply` in the record; `answer()` asks once more, `trace()` does not. Try `JSON_MODE=json_object` |

## Resources

- [OpenAI: create chat completion](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create):
  `usage`, the token counts a trace records
- [Python: `hashlib`](https://docs.python.org/3/library/hashlib.html): SHA-256 and `hexdigest()`
- [OpenTelemetry: traces](https://opentelemetry.io/docs/concepts/signals/traces/): how tracing tools record each step
  of a request as a span, the production form of these records
- [Joren et al., *Sufficient Context*, ICLR 2025](https://arxiv.org/abs/2411.06037): separating "the context was not
  enough" from "the model failed with enough context"

Next: [Lab 05](../05_Rebuild_With_LangChain/README.md), *How to rebuild a RAG pipeline with LangChain*. Back to [the lab list](../README.md).